# Example: BBBC021 — Mechanism of Action (MoA) Profiling

Perform morphological profiling on a multi-channel compound screen to analyze and predict the Mechanism of Action (MoA) of various treatments.

**Dataset:** [BBBC021](https://bbbc.broadinstitute.org/BBBC021) (3-channel fluorescent compound screen)

**Key Steps:**
1. **Metadata Integration:** Merge image metadata with compound MoA annotations.
2. **Multi-Channel Processing:** Link and process 3-channel images (Actin, Tubulin, DAPI).
3. **MoA Analysis:** Extract deep embeddings, compute extended properties, and classify compound effects within the phenospace.

#0. Google Colab installation
Skip this step if the notebook is not running on Google Colab.

In [ ]:
# @title Install PhenoMe (Colab only)
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"PhenoMe requires Python 3.12+. Detected Python {sys.version_info.major}.{sys.version_info.minor}."
    )

if IN_COLAB:
    # Install dependencies without overriding Colab's pre-installed torch/notebook stack
    %pip install -q --upgrade-strategy only-if-needed "numpy>=1.24" pandas "matplotlib>=3.7" plotly scikit-learn joblib tqdm tifffile h5py scipy "scikit-image>=0.26.0" ipywidgets anywidget pillow "umap-learn==0.5.12" "nibabel>=5.2.0,<6" "dcor==0.7"
    %pip install -q --no-deps phenome
    try:
        import cv2  # noqa: F401
    except ImportError:
        %pip install -q opencv-python-headless
    print("✓ PhenoMe installed successfully! Proceed to Step 1.")
else:
    print("✓ Running in local environment. Proceed to Step 1.")

In [ ]:
# @title Mount your Drive.
from google.colab import drive
drive.mount('/content/drive')

## Unzip the image data into Colab's working space
This step will take 2-5 minutes

In [ ]:
path2data = "/content/datasets/BBBC"
!mkdir -p $path2data
!unzip -q "/content/drive/MyDrive/IGNACIO ARGANDA/AITOR GONZALEZ/SPAOM2026/WORKSHOP/PHENOME_PRACTICAL_SPAOM2026/DATA/BBBC021/BBBC021_week123_filtered.zip" -d $path2data

## 1. Paths & Configuration

Define the paths to the BBBC021 dataset and the metadata files, including the Mechanism of Action (MoA) annotations.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

In [ ]:
IMAGE_DIR = "/content/datasets/BBBC/BBBC021_week123_filtered/images"
METADATA_PATH = "/content/datasets/BBBC/BBBC021_week123_filtered/metadata.csv"

## 2. Load & Merge Metadata

We load the image metadata and merge it with the MoA table to associate each treatment with its known biological mechanism.

In [ ]:
df_merged = pd.read_csv(METADATA_PATH)
df_merged.head()

## 3. Multi-Channel Data Discovery

BBBC021 consists of 3-channel images (Actin, Tubulin, DAPI). We specify the filename columns for each channel to ensure they are correctly linked and loaded. `channel_names` uses the same order so property columns are labelled by stain (e.g. `intensity_mean_Actin`) instead of `_ch0`.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    IMAGE_DIR,
    metadata_fn=make_dataframe_metadata_fn(
        df_merged,
        filename_column=["Image_FileName_Actin", "Image_FileName_Tubulin", "Image_FileName_DAPI"],
    ),
    channel_names=["Actin", "Tubulin", "DAPI"],
)
file_df.head()

## 4. Feature Extraction

We extract DINOv2 embeddings and compute an extended set of morphological properties to capture subtle phenotypic changes induced by the compounds.

> **Tip — Checkpoints & Resuming**:
> - **Save / Resume during extraction**: Pass `checkpoint_path="embeddings.h5"` to `process_images` and `checkpoint_path="properties.h5"` to `compute_properties`. Embeddings and properties are written incrementally to HDF5, saving RAM and enabling automatic resume if interrupted.
> - **Load precomputed checkpoints**: If checkpoints were already computed, skip feature extraction entirely by loading them with `pheno.load_embeddings(...)` and `pheno.load_properties(...)` (or `pheno.load_results(...)`).

In [ ]:
path2embeddings = "/content/drive/MyDrive/IGNACIO ARGANDA/AITOR GONZALEZ/SPAOM2026/WORKSHOP/PHENOME_PRACTICAL_SPAOM2026/DATA/BBBC021/checkpoints/BBBC021_checkpoint_DINOv2_embeddings.h5"
path2properties = "/content/drive/MyDrive/IGNACIO ARGANDA/AITOR GONZALEZ/SPAOM2026/WORKSHOP/PHENOME_PRACTICAL_SPAOM2026/DATA/BBBC021/checkpoints/BBBC021_checkpoint_properties.h5"

# Option A: Load precomputed checkpoints (skip extraction)
# pheno.load_embeddings("/path/to/bbbc021/.../embeddings.h5")
# pheno.load_properties("/path/to/bbbc021/.../properties.h5")

# Option B: Extract features (optionally add checkpoint_path to save/resume)
pheno.process_images(
    wrapper,
    batch_size=16,
    channel_mode="split",
    resize_size=1036,
    pad_size=1280,
    l2_normalize_channels=False,
    checkpoint_path= path2embeddings #"bbbc021_embeddings.h5",  # Save embeddings incrementally to HDF5
)

df_properties = pheno.compute_properties(
    property_preset="complete",
    n_jobs=4,
    checkpoint_path= path2properties # "bbbc021_properties.h5",  # Save properties incrementally to HDF5
)

## 5. Group Statistics

Aggregate morphological properties by compound and Mechanism of Action to identify characteristic phenotypic profiles.

In [ ]:
df_grouped = pheno.property_stats_by_group(
    group_by=["Image_Metadata_Compound", "Mechanism_of_action"],
    print_properties=["intensity_mean_Actin", "intensity_std_Actin"],
)

## 6. Visualization

Visualize the compound phenospace using t-SNE, colored by Mechanism of Action, to observe how different treatments cluster based on their morphological effects.

In [ ]:
pheno.plot_tsne(
    color_by="mechanism_of_action",
    n_components=2,
    exclude={"Image_metadata_compound": "taxol"},
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)

In [ ]:
pheno.plot_tsne(
    color_by="image_metadata_compound",
    n_components=2,
    filters={"mechanism_of_action": "Microtubule destabilizers"},
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)


## 7. Correlation Analysis
Compute correlations between morphological features and their embeddings to identify potential relationships.

In [ ]:
corr_results = pheno.compute_component_correlation(
    method="tsne",
    n_components=2,
    source="embeddings",
    exclude={"Image_Metadata_Compound": "taxol"},
    filters={"Mechanism_of_action": "Microtubule destabilizers"},
    top_k=5,
    correlation_method="pearson",
)

## 8. Interactive Exploration

Launch the interactive explorer to visually inspect the 3-channel images and explore the relationship between compound concentration and phenotypic response.

In [ ]:
# In Google Colab, use pheno.create_colab_interactive_explorer() instead.
e = pheno.create_interactive_explorer(
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)